<a href="https://colab.research.google.com/github/Sultoniromadhon/data-science-2026/blob/main/Pertemuan3_Sultoni_Romadhon_250401020198.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [130]:
# ============================================
# PIPELINE DATA CLEANING — HOUSING DATASET
# ============================================

import pandas as pd
import numpy as np
from scipy.stats.mstats import winsorize

# untuk API
import requests

In [131]:
# STEP 0 — Load & eksplorasi awal
df = pd.read_csv('/content/sample_data/housing_dirty.csv')

In [132]:
print('Shape awal:', df.shape)
print(df.isnull().sum())

Shape awal: (130, 7)
id               0
luas_m2         18
harga_juta      17
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [133]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB


,id,luas_m2,harga_juta,kamar,tahun_bangun
count,130.000000,112.000000,1.130000e+02,120.000000,130.000000
mean,65.500000,267.627679,8.856325e+05,3.433333,2062.638462
std,37.671829,885.664181,9.407144e+06,1.776283,701.684043
min,1.000000,-50.000000,-5.000000e+02,1.000000,1890.000000
25%,33.250000,87.050000,3.450000e+02,2.000000,1991.250000
50%,65.500000,193.800000,6.550000e+02,4.000000,2002.000000
75%,97.750000,280.675000,9.550000e+02,5.000000,2011.750000
max,130.000000,9500.000000,1.000000e+08,6.000000,9999.000000


In [134]:
# STEP 1 — Hapus Duplikat
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

Setelah hapus duplikat: (130, 7)


In [135]:
# STEP 2 — Normalisasi String
df['kota'] = df['kota'].astype(str).str.strip().str.title()
df['kondisi'] = df['kondisi'].astype(str).str.strip().str.lower()

In [136]:
# STEP 3 — Imputasi Missing Values
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

In [137]:
# STEP 4 — Tangani Outlier (IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

In [138]:
# STEP 5 — Validasi & Ekspor
print('Total missing:', df.isnull().sum().sum())
print('Total duplikat:', df.duplicated().sum())

Total missing: 0
Total duplikat: 0


In [139]:
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

In [140]:
print('Shape akhir:', df.shape)

Shape akhir: (130, 7)


In [141]:

df.to_csv('/content/sample_data/housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Dataset bersih tersimpan!


In [142]:
# Akses API JSONPlaceholder dan simpan respons sebagai DataFrame
response = requests.get("https://jsonplaceholder.typicode.com/users")
data = response.json()

df = pd.DataFrame(data)
df.head()

,id,name,username,email,address,phone,website,company
0,1,Leanne Graham,Bret,Sincere@april.biz,"{'street': 'Kulas Light', 'suite': 'Apt. 556',...",1-770-736-8031 x56442,hildegard.org,"{'name': 'Romaguera-Crona', 'catchPhrase': 'Mu..."
1,2,Ervin Howell,Antonette,Shanna@melissa.tv,"{'street': 'Victor Plains', 'suite': 'Suite 87...",010-692-6593 x09125,anastasia.net,"{'name': 'Deckow-Crist', 'catchPhrase': 'Proac..."
2,3,Clementine Bauch,Samantha,Nathan@yesenia.net,"{'street': 'Douglas Extension', 'suite': 'Suit...",1-463-123-4447,ramiro.info,"{'name': 'Romaguera-Jacobson', 'catchPhrase': ..."
3,4,Patricia Lebsack,Karianne,Julianne.OConner@kory.org,"{'street': 'Hoeger Mall', 'suite': 'Apt. 692',...",493-170-9623 x156,kale.biz,"{'name': 'Robel-Corkery', 'catchPhrase': 'Mult..."
4,5,Chelsey Dietrich,Kamren,Lucio_Hettinger@annie.ca,"{'street': 'Skiles Walks', 'suite': 'Suite 351...",(254)954-1289,demarco.info,"{'name': 'Keebler LLC', 'catchPhrase': 'User-c..."


In [143]:
url = "https://api.bmkg.go.id/publik/prakiraan-cuaca?adm4=31.71.01.1001"
data = requests.get(url).json()

cuaca = data["data"][0]["cuaca"]

flat = []
for group in cuaca:
    for item in group:
        flat.append(item)

df_bmkg = pd.DataFrame(flat)
df_bmkg.head()

,datetime,t,tcc,tp,weather,weather_desc,weather_desc_en,wd_deg,wd,wd_to,ws,hu,vs,vs_text,time_index,analysis_date,image,utc_datetime,local_datetime
0,2026-05-06T05:00:00Z,31,100,0.0,3,Berawan,Mostly Cloudy,65,NE,SW,6.9,63,11008,> 10 km,16-17,2026-05-05T12:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-05-06 05:00:00,2026-05-06 12:00:00
1,2026-05-06T08:00:00Z,30,100,0.2,3,Berawan,Mostly Cloudy,352,NW,SE,7.4,74,16291,> 10 km,19-20,2026-05-05T12:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-05-06 08:00:00,2026-05-06 15:00:00
2,2026-05-06T11:00:00Z,26,100,0.3,3,Berawan,Mostly Cloudy,226,SW,NE,3.9,88,6987,< 7 km,22-23,2026-05-05T12:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-05-06 11:00:00,2026-05-06 18:00:00
3,2026-05-06T14:00:00Z,25,99,0.3,3,Berawan,Mostly Cloudy,208,S,N,1.5,92,21898,> 10 km,25-26,2026-05-05T12:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-05-06 14:00:00,2026-05-06 21:00:00
4,2026-05-06T17:00:00Z,25,59,0.0,1,Cerah,Sunny,264,SW,NE,2.0,93,16639,> 10 km,28-29,2026-05-05T12:00:00,https://api-apps.bmkg.go.id/storage/icon/cuaca...,2026-05-06 17:00:00,2026-05-07 00:00:00
